# Analyse en Voorspellingen van Wedstrijden in de Jupiler Pro League

**Academisch project / Data-analyse notebook**

Auteur: Torben de Schrijver
Datum: [2026]

Beschrijving: Deze notebook is bedoeld om factoren te analyseren die voetbalwedstrijden beïnvloeden, en om voorspellingen van experts te vergelijken met datagedreven modellen.

## 0. Imports


In [428]:
import pyodbc
import pandas as pd
from datetime import datetime as dt
import os
import numpy as np

## 1. Introductie

### Doel van de analyse
Het doel van deze analyse is te onderzoeken hoe expertinschattingen van voetbalwedstrijden overeenkomen met data-gedreven voorspellingen op basis van historische wedstrijdstatistieken.

### Onderzoeksvraag
In welke mate zijn de voorspellingen van experts (journalisten, analisten, trainers) accuraat vergeleken met analyses gemaakt met programmeercode en machine learning modellen?

### Dataset beschrijving
De dataset bevat historische wedstrijddata van de Jupiler Pro League, inclusief:
- Wedstrijdresultaten
- Aantal doelpunten en kaarten
- Teamstatistieken (schoten, corners, ranglijstpositie, thuis/uit)
- Weersomstandigheden en stadioninformatie  
Daarnaast bevat de dataset ingevulde vragenlijsten van experts met hun voorspellingen en factorbeoordelingen.

### Overzicht van de methode
1. **Data verzamelen**: Historische wedstrijden en ingevulde vragenlijsten van experts.  
2. **Feature selectie**: Factoren zoals thuisvoordeel, recente vorm, rangverschil, weersomstandigheden.  
3. **Vergelijking voorspellingen**: Experts vs data-gedreven modellen.  
4. **Analyse**: Accuracy, bias-analyse, invloed van factoren en correlaties tussen expertinschattingen en resultaten.  
5. **Visualisatie en rapportage**: Grafieken, tabellen en samenvatting van inzichten.

## 2. Data laden

- Wedstrijddataset
- Weersdata
- Teams / Ranglijst
- Experts dataset

Commentaar over de structuur van de data

### Database connectie instellen

In [429]:
server = 'localhost'
database = 'JPL_Data'
# username = 'tds'
# password = 'tds'

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
#    f"UID={username};"
#    f"PWD={password};"
#    "Encrypt=yes;"
#    "TrustServerCertificate=yes;"
    "Trusted_Connection=yes;"
)
conn = pyodbc.connect(conn_str)

### Tabellen inladen

In [430]:
# Team
team_df = pd.read_sql("SELECT * FROM Team;", conn)

# Wedstrijd
wedstrijd_df = pd.read_sql("SELECT * FROM Wedstrijd;", conn)

# WedstrijdStatistiek
statistiek_df = pd.read_sql("SELECT * FROM WedstrijdStatistiek;", conn)

# WedstrijdWeer
weer_df = pd.read_sql("SELECT * FROM WedstrijdWeer;", conn)

# WedstrijdWeerVooraf
weer_vooraf_df = pd.read_sql("SELECT * FROM WedstrijdWeerVooraf;", conn)

C:\Users\desch\AppData\Local\Temp\ipykernel_129120\1562388262.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  team_df = pd.read_sql("SELECT * FROM Team;", conn)
C:\Users\desch\AppData\Local\Temp\ipykernel_129120\1562388262.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  wedstrijd_df = pd.read_sql("SELECT * FROM Wedstrijd;", conn)
C:\Users\desch\AppData\Local\Temp\ipykernel_129120\1562388262.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  statistiek_df = pd.read_sql("SELECT * FROM WedstrijdStatistiek;", conn)
C:\

### CSV (enquête) inladen

In [431]:
enquete_df = pd.read_csv('../data/experts_enquete.csv')

## 3. Data cleaning

### Wedstrijddata

In [432]:
wedstrijd_df.info()

wedstrijd_df = wedstrijd_df.drop(['divisie'], axis=1)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2006 entries, 0 to 2005
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   wedstrijd_id     2006 non-null   int64 
 1   divisie          2006 non-null   object
 2   wedstrijd_datum  2006 non-null   object
 3   wedstrijd_tijd   2006 non-null   object
 4   thuis_team       2006 non-null   object
 5   uit_team         2006 non-null   object
dtypes: int64(1), object(5)
memory usage: 94.2+ KB


In [433]:
print(statistiek_df.info())

# 1. Selecteer float64 kolommen
float_cols = statistiek_df.select_dtypes(include=['float64']).columns

# 2. Vul NaN met gemiddelde van elke kolom
statistiek_df[float_cols] = statistiek_df[float_cols].fillna(statistiek_df[float_cols].mean())

# 3. Rond af en converteer naar gewone int64
statistiek_df[float_cols] = statistiek_df[float_cols].round(0).astype('int')

# Check
print(statistiek_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2006 entries, 0 to 2005
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   wedstrijd_id               2006 non-null   int64  
 1   doelpunten_thuis_voltijd   2006 non-null   int64  
 2   doelpunten_uit_voltijd     2006 non-null   int64  
 3   resultaat_voltijd          2006 non-null   object 
 4   doelpunten_thuis_halftijd  2001 non-null   float64
 5   doelpunten_uit_halftijd    2001 non-null   float64
 6   resultaat_halftijd         2006 non-null   object 
 7   schoten_thuis              2000 non-null   float64
 8   schoten_uit                2000 non-null   float64
 9   schoten_op_doel_thuis      2000 non-null   float64
 10  schoten_op_doel_uit        2000 non-null   float64
 11  overtredingen_thuis        2000 non-null   float64
 12  overtredingen_uit          2000 non-null   float64
 13  corners_thuis              2000 non-null   float

In [434]:
team_df.info()

team_df = team_df.drop(['latitude', 'longitude'], axis=1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   team_naam           24 non-null     object 
 1   latitude            24 non-null     float64
 2   longitude           24 non-null     float64
 3   stadion_capaciteit  24 non-null     int64  
dtypes: float64(2), int64(1), object(1)
memory usage: 896.0+ bytes


In [435]:
weer_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2006 entries, 0 to 2005
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   wedstrijd_id                    2006 non-null   int64  
 1   temperatuur_c                   2006 non-null   float64
 2   neerslag_mm                     2006 non-null   float64
 3   relatieve_luchtvochtigheid_pct  2006 non-null   int64  
 4   windsnelheid_m_s                2006 non-null   float64
 5   windrichting_graden             2006 non-null   int64  
 6   windstoten_m_s                  2006 non-null   float64
 7   bewolking_pct                   2006 non-null   int64  
 8   weercode                        2006 non-null   int64  
 9   luchtdruk_hpa                   2006 non-null   float64
 10  temperatuur_gem_c               2006 non-null   float64
 11  neerslag_som_mm                 2006 non-null   float64
 12  windsnelheid_gem_m_s            20

In [436]:
weer_vooraf_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2006 entries, 0 to 2005
Data columns (total 6 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   wedstrijd_id                  2006 non-null   int64  
 1   temperatuur_gem_laatste48_c   2006 non-null   float64
 2   neerslag_som_laatste48_mm     2006 non-null   float64
 3   windstoten_max_laatste48_m_s  2006 non-null   float64
 4   regen_uren_laatste48          2006 non-null   int64  
 5   hitte_uren_laatste48          2006 non-null   int64  
dtypes: float64(3), int64(3)
memory usage: 94.2 KB


#### Tabellen mergen

In [437]:
wedstrijden_compleet_df = (
    wedstrijd_df
    .merge(statistiek_df, on='wedstrijd_id', how='left')
    .merge(weer_df, on='wedstrijd_id', how='left')
    .merge(weer_vooraf_df, on='wedstrijd_id', how='left')
)

# merge met team info (thuis en uit)
wedstrijden_compleet_df = (
    wedstrijden_compleet_df
    .merge(team_df.add_prefix('thuis_'), left_on='thuis_team', right_on='thuis_team_naam', how='left')
    .merge(team_df.add_prefix('uit_'), left_on='uit_team', right_on='uit_team_naam', how='left')
)

wedstrijden_compleet_df = wedstrijden_compleet_df.drop('wedstrijd_id', axis=1)

print("Shape van complete dataset:", wedstrijden_compleet_df.shape)
wedstrijden_compleet_df.info()
wedstrijden_compleet_df.head()

Shape van complete dataset: (2006, 44)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2006 entries, 0 to 2005
Data columns (total 44 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   wedstrijd_datum                 2006 non-null   object 
 1   wedstrijd_tijd                  2006 non-null   object 
 2   thuis_team                      2006 non-null   object 
 3   uit_team                        2006 non-null   object 
 4   doelpunten_thuis_voltijd        2006 non-null   int64  
 5   doelpunten_uit_voltijd          2006 non-null   int64  
 6   resultaat_voltijd               2006 non-null   object 
 7   doelpunten_thuis_halftijd       2006 non-null   int64  
 8   doelpunten_uit_halftijd         2006 non-null   int64  
 9   resultaat_halftijd              2006 non-null   object 
 10  schoten_thuis                   2006 non-null   int64  
 11  schoten_uit                     2006 non-null   int64  


,wedstrijd_datum,wedstrijd_tijd,thuis_team,uit_team,doelpunten_thuis_voltijd,doelpunten_uit_voltijd,resultaat_voltijd,doelpunten_thuis_halftijd,doelpunten_uit_halftijd,resultaat_halftijd,...,windstoten_max_m_s,temperatuur_gem_laatste48_c,neerslag_som_laatste48_mm,windstoten_max_laatste48_m_s,regen_uren_laatste48,hitte_uren_laatste48,thuis_team_naam,thuis_stadion_capaciteit,uit_team_naam,uit_stadion_capaciteit
0,2019-07-26,19:30:00,Genk,Kortrijk,2,1,H,0,1,A,...,35.6,30.6,0.0,37.4,0,23,Genk,23718,Kortrijk,9399
1,2019-07-27,17:00:00,Cercle Brugge,Standard,0,2,A,0,0,D,...,28.1,25.9,0.0,45.4,0,27,Cercle Brugge,29062,Standard,27670
2,2019-07-27,19:00:00,St Truiden,Mouscron,0,1,A,0,1,A,...,28.1,29.0,0.3,31.3,1,27,St Truiden,14600,Mouscron,12800
3,2019-07-27,19:00:00,Waregem,Mechelen,0,2,A,0,1,A,...,21.2,25.8,0.1,38.9,1,26,Waregem,12414,Mechelen,16672
4,2019-07-27,19:30:00,Waasland-Beveren,Club Brugge,1,3,A,1,1,D,...,24.5,26.8,0.0,38.5,0,25,Waasland-Beveren,8190,Club Brugge,29062


### Enquête

In [438]:
# 1-5 schaalvragen
scale_1_5 = [
    "Hoe belangrijk is het thuisvoordeel bij het voorspellen van winst/verlies?",
    "In welke mate beïnvloedt stadioncapaciteit het totaal aantal schoten in een wedstrijd?  ",
    "In welke mate beïnvloedt stadioncapaciteit het aantal overtredingen van het uitteam?  ",
    "Heeft regen invloed op het aantal doelpunten?  ",
    "In welke mate verwacht u dat regen het aantal kaarten in een wedstrijd beïnvloedt? ",
    "Hoe bepalend is recente vorm (laatste 5 wedstrijden) voor het resultaat? ",
    "Hoe belangrijk is het verschil in het klassement voor het voorspellen van een wedstrijd? ",
    "In welke mate hebben eerdere confrontaties invloed in uw voorspelling? ",
    "In welke mate verwacht u dat een hoger aantal schoten samenhangt met meer corners tijdens een wedstrijd? ",
    "Hoe zeker bent u van deze match voorspelling?",
    "Hoeveel invloed heeft een derby/rivalen match op het aantal kaarten?",
    "Hoeveel invloed heeft een wedstrijd voor de titel op het aantal schoten?",
    "Hoeveel invloed heeft een wedstrijd voor degradatie op het aantal kaarten?"
]

# percentage vragen
scale_0_100 = [
    "Kans op thuiswinst  (0 - 100)",
    "Kans op gelijkspel  (0 - 100)",
    "Kans op uitwinst  (0 - 100)",
    "Als team A 10 plaatsen boven team B staat in het klassement: hoe groot is volgens u de kans op een overwinning van team A?  Percentage (0 - 100)",
    "Stel dat een team 3 keer op rij verloren heeft, hoe groot is volgens u de kans op een 4e nederlaag?  Percentage (0 - 100)"
]

# ===============================
# 2. Controle 1-5 schaal
# ===============================

for col in scale_1_5:
    if col in enquete_df.columns:
        enquete_df[col] = pd.to_numeric(enquete_df[col], errors="coerce")

        fouten = enquete_df[
            (enquete_df[col] < 1) |
            (enquete_df[col] > 5) |
            (enquete_df[col] % 1 != 0)
        ]

        if len(fouten) > 0:
            print(f"Foutieve waarden in kolom: {col}")
            print(fouten[[col]])

# ===============================
# 3. Controle 0-100 schaal
# ===============================

for col in scale_0_100:
    if col in enquete_df.columns:
        enquete_df[col] = pd.to_numeric(enquete_df[col], errors="coerce")

        fouten = enquete_df[
            (enquete_df[col] < 0) |
            (enquete_df[col] > 100) |
            (enquete_df[col] % 1 != 0)
        ]

        if len(fouten) > 0:
            print(f"Foutieve waarden in kolom: {col}")
            print(fouten[[col]])

enquete_df["kans_som_match1"] = (
    enquete_df["Kans op thuiswinst  (0 - 100)"] +
    enquete_df["Kans op gelijkspel  (0 - 100)"] +
    enquete_df["Kans op uitwinst  (0 - 100)"]
)

fouten = enquete_df[enquete_df["kans_som_match1"] != 100]
print(fouten)

Empty DataFrame
Columns: [Tijdstempel, Hoe belangrijk is het thuisvoordeel bij het voorspellen van winst/verlies?, In welke mate beïnvloedt stadioncapaciteit het totaal aantal schoten in een wedstrijd?  , In welke mate beïnvloedt stadioncapaciteit het aantal overtredingen van het uitteam?  , Heeft regen invloed op het aantal doelpunten?  , In welke mate verwacht u dat regen het aantal kaarten in een wedstrijd beïnvloedt? , Hoe bepalend is recente vorm (laatste 5 wedstrijden) voor het resultaat? , Hoe belangrijk is het verschil in het klassement voor het voorspellen van een wedstrijd? , In welke mate hebben eerdere confrontaties invloed in uw voorspelling? , In welke mate verwacht u dat een hoger aantal schoten samenhangt met meer corners tijdens een wedstrijd? , Kans op thuiswinst  (0 - 100), Kans op gelijkspel  (0 - 100), Kans op uitwinst  (0 - 100), Verwacht aantal doelpunten thuisteam?, Verwacht aantal doelpunten uitteam?, Hoe zeker bent u van deze match voorspelling?, Kans op thuis

In [439]:
enquete_df.info()

enquete_df = enquete_df.drop(['Tijdstempel', 'kans_som_match1'], axis=1)

enquete_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 50 columns):
 #   Column                                                                                                                                            Non-Null Count  Dtype 
---  ------                                                                                                                                            --------------  ----- 
 0   Tijdstempel                                                                                                                                       5 non-null      object
 1   Hoe belangrijk is het thuisvoordeel bij het voorspellen van winst/verlies?                                                                        5 non-null      int64 
 2   In welke mate beïnvloedt stadioncapaciteit het totaal aantal schoten in een wedstrijd?                                                            5 non-null      int64 
 3   In welke mate beïnvloedt

,Hoe belangrijk is het thuisvoordeel bij het voorspellen van winst/verlies?,In welke mate beïnvloedt stadioncapaciteit het totaal aantal schoten in een wedstrijd?,In welke mate beïnvloedt stadioncapaciteit het aantal overtredingen van het uitteam?,Heeft regen invloed op het aantal doelpunten?,In welke mate verwacht u dat regen het aantal kaarten in een wedstrijd beïnvloedt?,Hoe bepalend is recente vorm (laatste 5 wedstrijden) voor het resultaat?,Hoe belangrijk is het verschil in het klassement voor het voorspellen van een wedstrijd?,In welke mate hebben eerdere confrontaties invloed in uw voorspelling?,In welke mate verwacht u dat een hoger aantal schoten samenhangt met meer corners tijdens een wedstrijd?,Kans op thuiswinst (0 - 100),...,Rangschik de onderstaande factoren van meeste invloed → minste invloed voor aantal kaarten [5e],Rangschik de onderstaande factoren van meeste invloed → minste invloed voor aantal kaarten [6e],"Als het hard regent, zullen er ... kaarten verwacht worden.","Als het hard regent, zullen er ... doelpunten verwacht worden.",Als het thuisteam een aanvallende speelstijl hanteert en het uitteam defensief speelt zullen er gemiddeld ... doelpunten vallen.,"Als het erg warm (>23°C) is, zal het aantal schoten ... zijn dan gemiddeld.","Als er een derby is, of rivalen spelen, zal het aantal schoten van beide teams ... zijn dan gemiddeld.",Hoeveel invloed heeft een derby/rivalen match op het aantal kaarten?,Hoeveel invloed heeft een wedstrijd voor de titel op het aantal schoten?,Hoeveel invloed heeft een wedstrijd voor degradatie op het aantal kaarten?
0,4,1,1,3,2,4,4,4,5,50,...,stadioncapaciteit,tijdstip,meer,zelfde,meer,zelfde,meer,4,4,4
1,3,2,2,3,2,4,4,4,4,52,...,stadioncapaciteit,Recente vorm,meer,zelfde,zelfde,minder,meer,5,4,4
2,4,3,3,2,1,4,4,3,1,65,...,neerslag,tijdstip,zelfde,zelfde,meer,zelfde,minder,4,4,2
3,4,4,5,1,3,3,3,4,4,60,...,neerslag,stadioncapaciteit,meer,zelfde,minder,minder,meer,5,5,5
4,5,3,1,3,1,1,5,4,1,70,...,derby/rivalen,tijdstip,zelfde,zelfde,zelfde,zelfde,zelfde,4,1,1


## 4. Feature engineering

In [440]:
# Kopieer je dataframe voor veiligheid
df = wedstrijden_compleet_df.copy()

### Seizoenkolom toevoegen

In [441]:
def seizoen_bepalen(datum):
    jaar = datum.year
    if datum.month >= 7:  # juli-december → nieuw seizoen startjaar
        return f"{jaar}/{jaar+1}"
    else:                # januari-juni → vorig jaar als startjaar
        return f"{jaar-1}/{jaar}"

# Voeg kolom toe
df['seizoen'] = df['wedstrijd_datum'].apply(seizoen_bepalen)

# Controleer relevante kolommen
df[['wedstrijd_datum', 'wedstrijd_tijd', 'thuis_team', 'uit_team', 'resultaat_voltijd', 'seizoen']].head()

,wedstrijd_datum,wedstrijd_tijd,thuis_team,uit_team,resultaat_voltijd,seizoen
0,2019-07-26,19:30:00,Genk,Kortrijk,H,2019/2020
1,2019-07-27,17:00:00,Cercle Brugge,Standard,A,2019/2020
2,2019-07-27,19:00:00,St Truiden,Mouscron,A,2019/2020
3,2019-07-27,19:00:00,Waregem,Mechelen,A,2019/2020
4,2019-07-27,19:30:00,Waasland-Beveren,Club Brugge,A,2019/2020


### Rangschikking

In [442]:
# Sorteer op datum per seizoen
df = df.sort_values(['seizoen','wedstrijd_datum'])

# Kolommen voor rang
df['thuis_rank'] = 0
df['uit_rank'] = 0

# Itereer per seizoen
for seizoen in df['seizoen'].unique():
    df_seizoen = df[df['seizoen']==seizoen].copy()
    teams = pd.unique(df_seizoen[['thuis_team','uit_team']].values.ravel())
    
    # Initialiseer dict
    stats_dict = {team: {'punten':0, 'doelpunten_voor':0, 'doelpunten_tegen':0} for team in teams}
    
    ranglijst_thuis = []
    ranglijst_uit = []
    
    for idx, row in df_seizoen.iterrows():
        thuis = row['thuis_team']
        uit = row['uit_team']
        resultaat = row['resultaat_voltijd']
        
        # Punten toevoegen
        if resultaat == 'H':
            stats_dict[thuis]['punten'] += 3
        elif resultaat == 'A':
            stats_dict[uit]['punten'] += 3
        elif resultaat == 'D':
            stats_dict[thuis]['punten'] += 1
            stats_dict[uit]['punten'] += 1
        
        # Doelpunten bijwerken
        doel_thuis = row['doelpunten_thuis_voltijd'] if pd.notnull(row['doelpunten_thuis_voltijd']) else 0
        doel_uit = row['doelpunten_uit_voltijd'] if pd.notnull(row['doelpunten_uit_voltijd']) else 0
        
        stats_dict[thuis]['doelpunten_voor'] += doel_thuis
        stats_dict[thuis]['doelpunten_tegen'] += doel_uit
        
        stats_dict[uit]['doelpunten_voor'] += doel_uit
        stats_dict[uit]['doelpunten_tegen'] += doel_thuis
        
        # Bereken doelpuntenverschil
        for team in teams:
            stats_dict[team]['doelpuntenverschil'] = stats_dict[team]['doelpunten_voor'] - stats_dict[team]['doelpunten_tegen']
        
        # Sorteer op punten, dan doelpuntenverschil
        sorted_teams = sorted(stats_dict.items(), key=lambda x: (-x[1]['punten'], -x[1]['doelpuntenverschil'], x[0]))
        team_rank = {team:rank+1 for rank, (team, stats) in enumerate(sorted_teams)}
        
        ranglijst_thuis.append(team_rank[thuis])
        ranglijst_uit.append(team_rank[uit])
    
    df.loc[df['seizoen']==seizoen, 'thuis_rank'] = ranglijst_thuis
    df.loc[df['seizoen']==seizoen, 'uit_rank'] = ranglijst_uit

# Check
df[['wedstrijd_datum','thuis_team','uit_team','resultaat_voltijd','seizoen','thuis_rank','uit_rank']].tail(15)

,wedstrijd_datum,thuis_team,uit_team,resultaat_voltijd,seizoen,thuis_rank,uit_rank
1991,2026-03-07,Oud-Heverlee Leuven,Westerlo,A,2025/2026,14,7
1992,2026-03-07,Dender,Charleroi,D,2025/2026,16,11
1993,2026-03-07,St. Gilloise,Genk,H,2025/2026,1,6
1994,2026-03-08,Club Brugge,Anderlecht,D,2025/2026,2,4
1995,2026-03-08,Gent,Mechelen,H,2025/2026,6,5
1996,2026-03-08,Waregem,Standard,A,2025/2026,12,9
1997,2026-03-08,St Truiden,Cercle Brugge,H,2025/2026,3,13
1998,2026-03-13,Gent,Waregem,H,2025/2026,5,12
1999,2026-03-14,Charleroi,Oud-Heverlee Leuven,A,2025/2026,11,12
2000,2026-03-14,St. Gilloise,Dender,H,2025/2026,1,16


### Titelstrijd / degradatiestrijd

In [443]:
# Voeg velden toe
df['titel_match'] = 0
df['degradatie_match'] = 0

# Sorteer op datum per seizoen
df = df.sort_values(['seizoen', 'wedstrijd_datum'])

for seizoen in df['seizoen'].unique():
    df_seizoen = df[df['seizoen'] == seizoen].copy()
    teams = pd.unique(df_seizoen[['thuis_team', 'uit_team']].values.ravel())
    
    # Initialiseer stats
    stats_dict = {team: {'punten':0, 'doelpunten_voor':0, 'doelpunten_tegen':0} for team in teams}
    
    for idx, row in df_seizoen.iterrows():
        thuis = row['thuis_team']
        uit = row['uit_team']
        resultaat = row['resultaat_voltijd']
        
        # Punten
        if resultaat == 'H':
            stats_dict[thuis]['punten'] += 3
        elif resultaat == 'A':
            stats_dict[uit]['punten'] += 3
        elif resultaat == 'D':
            stats_dict[thuis]['punten'] += 1
            stats_dict[uit]['punten'] += 1
        
        # Doelpunten
        doel_thuis = row['doelpunten_thuis_voltijd'] if pd.notnull(row['doelpunten_thuis_voltijd']) else 0
        doel_uit = row['doelpunten_uit_voltijd'] if pd.notnull(row['doelpunten_uit_voltijd']) else 0
        stats_dict[thuis]['doelpunten_voor'] += doel_thuis
        stats_dict[thuis]['doelpunten_tegen'] += doel_uit
        stats_dict[uit]['doelpunten_voor'] += doel_uit
        stats_dict[uit]['doelpunten_tegen'] += doel_thuis
        
        # Doelpuntenverschil
        for team in teams:
            stats_dict[team]['doelpuntenverschil'] = stats_dict[team]['doelpunten_voor'] - stats_dict[team]['doelpunten_tegen']
        
        # Ranglijst op punten en doelpuntenverschil
        sorted_teams = sorted(stats_dict.items(), key=lambda x: (-x[1]['punten'], -x[1]['doelpuntenverschil'], x[0]))
        team_rank = {team: rank+1 for rank, (team, stats) in enumerate(sorted_teams)}
        
        thuis_rank = team_rank[thuis]
        uit_rank = team_rank[uit]
        punten_thuis = stats_dict[thuis]['punten']
        punten_uit = stats_dict[uit]['punten']
        
        # Titelwedstrijd: beide in top2, >40 punten, max 5 punten verschil
        if thuis_rank <= 2 and uit_rank <= 2 and punten_thuis > 40 and punten_uit > 40 and abs(punten_thuis - punten_uit) <= 5:
            df.loc[idx, 'titel_match'] = 1
        
        # Degradatiewedstrijd: seizoen >20 wedstrijden, beide in laatste 3, max 3 punten verschil
        if idx > 20:  # indicatief: seizoen gevorderd
            max_rank = len(teams)
            if thuis_rank >= max_rank-2 and uit_rank >= max_rank-2 and abs(punten_thuis - punten_uit) <= 3:
                df.loc[idx, 'degradatie_match'] = 1

In [444]:
# Toon enkele wedstrijden voor titelstrijd
titel_wedstrijden = df[df['titel_match'] == 1]
print("Voorbeeld Titelwedstrijden:")
print(titel_wedstrijden[['wedstrijd_datum', 'wedstrijd_tijd','thuis_team','uit_team','thuis_rank','uit_rank','titel_match']].head(10))

# Toon enkele wedstrijden voor degradatiestrijd
degradatie_wedstrijden = df[df['degradatie_match'] == 1]
print("\nVoorbeeld Degradatiewedstrijden:")
print(degradatie_wedstrijden[['wedstrijd_datum','wedstrijd_tijd','thuis_team','uit_team','thuis_rank','uit_rank','degradatie_match']].head(10))

Voorbeeld Titelwedstrijden:
     wedstrijd_datum wedstrijd_tijd    thuis_team      uit_team  thuis_rank  \
1103      2023-03-12       17:30:00          Genk  St. Gilloise           1   
1410      2024-04-14       17:30:00    Anderlecht  St. Gilloise           2   
1440      2024-05-05       17:30:00  St. Gilloise    Anderlecht           1   

      uit_rank  titel_match  
1103         2            1  
1410         1            1  
1440         2            1  

Voorbeeld Degradatiewedstrijden:
     wedstrijd_datum wedstrijd_tijd           thuis_team          uit_team  \
122       2019-11-23       19:00:00     Waasland-Beveren     Cercle Brugge   
169       2020-01-18       19:00:00             Oostende  Waasland-Beveren   
354       2020-12-01       16:00:00             Mouscron        St Truiden   
421       2021-01-19       17:45:00             Mouscron  Waasland-Beveren   
476       2021-02-20       17:30:00             Mouscron     Cercle Brugge   
880       2022-08-19       19:45:

### Rivalen / Derby

In [445]:
# Lijst van echte Belgische derby's / klassieke rivalen
klassieke_rivalen = [
    ("Antwerp", "Beerschot VA"),            # Antwerpse derby
    ("RSC Anderlecht", "Club Brugge"),      # De Topper / Belgische klassieker
    ("Club Brugge", "Cercle Brugge"),       # Brugse derby
    ("Club Brugge", "Union Saint-Gilloise"),# Moderne rivaliteit
    ("Standard", "RSC Anderlecht")          # Belgian Clasico
]

# Functie om te controleren of een wedstrijd een derby/rivaliteit is
def is_derby(row):
    thuis = row['thuis_team']
    uit = row['uit_team']
    return int(any((thuis, uit) == pair or (uit, thuis) == pair for pair in klassieke_rivalen))

# Voeg het veld toe aan je dataframe
wedstrijden_compleet_df['derby/rivaal'] = wedstrijden_compleet_df.apply(is_derby, axis=1)

# Check een paar voorbeelden
print(wedstrijden_compleet_df[['thuis_team', 'uit_team', 'derby/rivaal']].head(10))

         thuis_team     uit_team  derby/rivaal
0              Genk     Kortrijk             0
1     Cercle Brugge     Standard             0
2        St Truiden     Mouscron             0
3           Waregem     Mechelen             0
4  Waasland-Beveren  Club Brugge             0
5        Anderlecht     Oostende             0
6         Charleroi         Gent             0
7             Eupen      Antwerp             0
8       Club Brugge   St Truiden             0
9          Standard      Waregem             0


### Aanvallende of Verdedigende ploeg

In [446]:
# ===============================
# Aanvallend vs Verdedigend team feature
# ===============================

# 1. Team statistieken berekenen (thuis + uit gecombineerd)
home_stats = df.groupby("thuis_team").agg({
    "schoten_thuis": "mean",
    "doelpunten_thuis_voltijd": "mean",
    "corners_thuis": "mean",
    "schoten_uit": "mean",
    "doelpunten_uit_voltijd": "mean"
}).rename(columns={
    "schoten_thuis": "shots_for",
    "doelpunten_thuis_voltijd": "goals_for",
    "corners_thuis": "corners_for",
    "schoten_uit": "shots_against",
    "doelpunten_uit_voltijd": "goals_against"
})

away_stats = df.groupby("uit_team").agg({
    "schoten_uit": "mean",
    "doelpunten_uit_voltijd": "mean",
    "corners_uit": "mean",
    "schoten_thuis": "mean",
    "doelpunten_thuis_voltijd": "mean"
}).rename(columns={
    "schoten_uit": "shots_for",
    "doelpunten_uit_voltijd": "goals_for",
    "corners_uit": "corners_for",
    "schoten_thuis": "shots_against",
    "doelpunten_thuis_voltijd": "goals_against"
})

# combineer thuis en uit statistieken
team_stats = pd.concat([home_stats, away_stats]).groupby(level=0).mean()

# 2. Attack en Defense score berekenen
team_stats["attack_score"] = (
    0.5 * team_stats["shots_for"] +
    0.3 * team_stats["goals_for"] +
    0.2 * team_stats["corners_for"]
)

team_stats["defense_score"] = (
    0.7 * team_stats["goals_against"] +
    0.3 * team_stats["shots_against"]
)

# 3. Style score
team_stats["style_score"] = team_stats["attack_score"] - team_stats["defense_score"]

# 4. Dynamische classificatie via percentielen
low_threshold = team_stats["style_score"].quantile(0.33)
high_threshold = team_stats["style_score"].quantile(0.66)

def classify_style(x):
    if x >= high_threshold:
        return "Aanvallend"
    elif x <= low_threshold:
        return "Verdedigend"
    else:
        return "Gebalanceerd"

team_stats["speelstijl"] = team_stats["style_score"].apply(classify_style)

# 5. Speelstijl toevoegen aan wedstrijd dataset
df = df.merge(team_stats["speelstijl"], left_on="thuis_team", right_index=True, how="left")
df = df.rename(columns={"speelstijl": "thuis_speelstijl"})

df = df.merge(team_stats["speelstijl"], left_on="uit_team", right_index=True, how="left")
df = df.rename(columns={"speelstijl": "uit_speelstijl"})

# 6. Controle
print(team_stats[["attack_score","defense_score","style_score","speelstijl"]].sort_values("style_score"))


# Controle van dataframe
df[["thuis_team","uit_team","thuis_speelstijl","uit_speelstijl"]].head()

                     attack_score  defense_score  style_score    speelstijl
Seraing                  5.598529       5.480882     0.117647   Verdedigend
RWD Molenbeek            6.113889       5.944444     0.169444   Verdedigend
Waasland-Beveren         5.758720       5.217742     0.540978   Verdedigend
Dender                   6.428277       5.712185     0.716092   Verdedigend
Beerschot VA             6.956731       5.956731     1.000000   Verdedigend
Eupen                    6.670876       5.551520     1.119356   Verdedigend
Waregem                  6.770777       5.406196     1.364580   Verdedigend
Mouscron                 5.876764       4.468044     1.408720   Verdedigend
Kortrijk                 6.868006       5.328300     1.539706  Gebalanceerd
RAAL La Louviere         6.218333       4.520000     1.698333  Gebalanceerd
Oud-Heverlee Leuven      7.558702       5.494874     2.063827  Gebalanceerd
Oostende                 7.062821       4.882051     2.180769  Gebalanceerd
Standard    

,thuis_team,uit_team,thuis_speelstijl,uit_speelstijl
0,Genk,Kortrijk,Aanvallend,Gebalanceerd
1,Cercle Brugge,Standard,Aanvallend,Gebalanceerd
2,St Truiden,Mouscron,Gebalanceerd,Verdedigend
3,Waregem,Mechelen,Verdedigend,Gebalanceerd
4,Waasland-Beveren,Club Brugge,Verdedigend,Aanvallend


### Recente Vorm (laatste 5 wedstrijden)

In [447]:
df['thuis_form_5'] = 0
df['uit_form_5'] = 0

for seizoen in df['seizoen'].unique():
    df_seizoen = df[df['seizoen'] == seizoen].copy()
    teams = pd.unique(df_seizoen[['thuis_team','uit_team']].values.ravel())
    form_dict = {team: [] for team in teams}  # lijst van laatste resultaten (3/1/0)

    thuis_form = []
    uit_form = []

    for idx, row in df_seizoen.iterrows():
        thuis = row['thuis_team']
        uit = row['uit_team']
        resultaat = row['resultaat_voltijd']

        # recente vorm berekenen
        if len(form_dict[thuis]) >= 5:
            thuis_form.append(np.mean(form_dict[thuis][-5:]))
        else:
            thuis_form.append(np.mean(form_dict[thuis]) if form_dict[thuis] else 0)

        if len(form_dict[uit]) >= 5:
            uit_form.append(np.mean(form_dict[uit][-5:]))
        else:
            uit_form.append(np.mean(form_dict[uit]) if form_dict[uit] else 0)

        # score toevoegen
        if resultaat == 'H':
            form_dict[thuis].append(3)
            form_dict[uit].append(0)
        elif resultaat == 'A':
            form_dict[thuis].append(0)
            form_dict[uit].append(3)
        else:  # gelijkspel
            form_dict[thuis].append(1)
            form_dict[uit].append(1)

    df.loc[df['seizoen']==seizoen,'thuis_form_5'] = thuis_form
    df.loc[df['seizoen']==seizoen,'uit_form_5'] = uit_form

C:\Users\desch\AppData\Local\Temp\ipykernel_129120\377792982.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0, 0, 0, 0, 0, 0, 0, 0, np.float64(3.0), np.float64(3.0), np.float64(0.0), np.float64(3.0), np.float64(3.0), np.float64(1.0), np.float64(3.0), np.float64(3.0), np.float64(0.5), np.float64(3.0), np.float64(0.0), np.float64(0.0), np.float64(1.5), np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(1.3333333333333333), np.float64(2.3333333333333335), np.float64(0.3333333333333333), np.float64(2.0), np.float64(2.0), np.float64(1.3333333333333333), np.float64(1.0), np.float64(1.5), np.float64(0.0), np.float64(0.75), np.float64(1.5), np.float64(2.25), np.float64(1.75), np.float64(2.0), np.float64(0.4), np.float64(1.4), np.float64(0.2), np.float64(2.5), np.float64(0.4), np.float64(1.75), np.float64(1.4), np.float64(1.6), np.float64(1.4), np.float64(0.4), np.float64(1.4), 

### H2H

In [448]:
# Aantal overwinningen thuisteam tegen uitteam in verleden
def calc_h2h(df):
    h2h_wins = []
    for idx, row in df.iterrows():
        thuis = row['thuis_team']
        uit = row['uit_team']
        datum = row['wedstrijd_datum']
        h2h_past = df[(df['thuis_team']==thuis) & (df['uit_team']==uit) & (df['wedstrijd_datum'] < datum)]
        wins = np.sum(h2h_past['resultaat_voltijd']=='H')
        h2h_wins.append(wins)
    return h2h_wins

df['h2h_thuis_wins'] = calc_h2h(df)

### Tijdstip

In [449]:
# Zorg dat 'wedstrijd_datum' datetime is
df['wedstrijd_datum'] = pd.to_datetime(df['wedstrijd_datum'], errors='coerce')

# Dag van de week, 0=maandag, 6=zondag
df['weekday'] = df['wedstrijd_datum'].dt.weekday

# Avondmatch indicator (18:00+) indien tijd beschikbaar
if 'wedstrijd_tijd' in df.columns:
    # Zorg dat tijd ook datetime is
    df['avond_match'] = df['wedstrijd_tijd'].apply(lambda t: 1 if pd.notnull(t) and t.hour >= 18 else 0)
else:
    df['avond_match'] = 0

### Extreme weersomstandigheden

In [450]:
# regen > 2 mm = harde regen, temperatuur >23°C = warm
df['regen'] = df['neerslag_som_mm'].apply(lambda x: 1 if pd.notnull(x) and x>0 else 0)
df['harde_regen'] = df['neerslag_mm'].apply(lambda x: 1 if pd.notnull(x) and x>2 else 0)
df['warm'] = df['temperatuur_gem_c'].apply(lambda x: 1 if pd.notnull(x) and x>23 else 0)

In [451]:
df[df['regen'] > 0]

,wedstrijd_datum,wedstrijd_tijd,thuis_team,uit_team,doelpunten_thuis_voltijd,doelpunten_uit_voltijd,resultaat_voltijd,doelpunten_thuis_halftijd,doelpunten_uit_halftijd,resultaat_halftijd,...,thuis_speelstijl,uit_speelstijl,thuis_form_5,uit_form_5,h2h_thuis_wins,weekday,avond_match,regen,harde_regen,warm
1,2019-07-27,17:00:00,Cercle Brugge,Standard,0,2,A,0,0,D,...,Aanvallend,Gebalanceerd,0.0,0.0,0,5,0,1,0,0
2,2019-07-27,19:00:00,St Truiden,Mouscron,0,1,A,0,1,A,...,Gebalanceerd,Verdedigend,0.0,0.0,0,5,1,1,1,0
3,2019-07-27,19:00:00,Waregem,Mechelen,0,2,A,0,1,A,...,Verdedigend,Gebalanceerd,0.0,0.0,0,5,1,1,0,0
4,2019-07-27,19:30:00,Waasland-Beveren,Club Brugge,1,3,A,1,1,D,...,Verdedigend,Aanvallend,0.0,0.0,0,5,1,1,0,0
5,2019-07-28,13:30:00,Anderlecht,Oostende,1,2,A,1,1,D,...,Aanvallend,Gebalanceerd,0.0,0.0,0,6,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1984,2026-02-28,17:15:00,Antwerp,St Truiden,1,0,H,0,0,D,...,Aanvallend,Gebalanceerd,0.6,2.4,4,5,0,1,0,0
1998,2026-03-13,19:45:00,Gent,Waregem,2,0,H,1,0,H,...,Aanvallend,Verdedigend,1.2,0.6,3,4,1,1,0,0
1999,2026-03-14,15:00:00,Charleroi,Oud-Heverlee Leuven,0,2,A,0,0,D,...,Aanvallend,Gebalanceerd,0.2,1.2,1,5,0,1,0,0
2000,2026-03-14,17:15:00,St. Gilloise,Dender,2,0,H,0,0,D,...,Aanvallend,Verdedigend,2.2,0.4,1,5,0,1,0,0


### Speelstijl-combinaties

In [452]:
# 1=aanvallend, 0=gebalanceerd, -1=verdedigend
# al aanwezig in df als 'speelstijl' per team
# Maak combinaties:
def speelstijl_combo(row):
    if row['thuis_speelstijl']==1 and row['uit_speelstijl']==-1:
        return "thuis_aanvallend_uit_verdedigend"
    elif row['thuis_speelstijl']==-1 and row['uit_speelstijl']==1:
        return "thuis_verdedigend_uit_aanvallend"
    else:
        return "gebalanceerd"

df['speelstijl_combo'] = df.apply(speelstijl_combo, axis=1)

### Weekdag en Weekend

In [453]:
# Weekend indicator: 1 als zaterdag (5) of zondag (6), anders 0
df['is_weekend'] = df['weekday'].apply(lambda x: 1 if x >= 5 else 0)

# Controle
df[['wedstrijd_datum', 'weekday', 'is_weekend']].head(10)

,wedstrijd_datum,weekday,is_weekend
0,2019-07-26,4,0
1,2019-07-27,5,1
2,2019-07-27,5,1
3,2019-07-27,5,1
4,2019-07-27,5,1
5,2019-07-28,6,1
6,2019-07-28,6,1
7,2019-07-28,6,1
8,2019-08-02,4,0
9,2019-08-03,5,1


### Rankverschil en vormverschil

In [454]:
# Zorg dat de kolommen numeriek zijn
df['thuis_rank'] = pd.to_numeric(df['thuis_rank'], errors='coerce')
df['uit_rank'] = pd.to_numeric(df['uit_rank'], errors='coerce')
df['thuis_form_5'] = pd.to_numeric(df['thuis_form_5'], errors='coerce')
df['uit_form_5'] = pd.to_numeric(df['uit_form_5'], errors='coerce')

# Rank verschil: positief → thuisteam hoger gerankt
df['rank_diff'] = df['thuis_rank'] - df['uit_rank']

# Vorm verschil: positief → thuisteam recent beter
df['form_diff'] = df['thuis_form_5'] - df['uit_form_5']

# Controle: eerste 10 rijen
df[['thuis_team','uit_team','thuis_rank','uit_rank','rank_diff',
    'thuis_form_5','uit_form_5','form_diff']].tail(10)

,thuis_team,uit_team,thuis_rank,uit_rank,rank_diff,thuis_form_5,uit_form_5,form_diff
1996,Waregem,Standard,12,9,3,0.6,1.6,-1.0
1997,St Truiden,Cercle Brugge,3,13,-10,1.8,1.4,0.4
1998,Gent,Waregem,5,12,-7,1.2,0.6,0.6
1999,Charleroi,Oud-Heverlee Leuven,11,12,-1,0.2,1.2,-1.0
2000,St. Gilloise,Dender,1,16,-15,2.2,0.4,1.8
2001,Westerlo,Club Brugge,8,2,6,2.0,2.6,-0.6
2002,Genk,St Truiden,7,3,4,1.8,2.4,-0.6
2003,Antwerp,Standard,10,8,2,0.8,1.6,-0.8
2004,Mechelen,Anderlecht,4,5,-1,1.8,1.6,0.2
2005,Cercle Brugge,RAAL La Louviere,15,13,2,1.4,0.6,0.8


### Alle features

In [455]:
# Print alle kolommen van je dataframe
print("Aantal kolommen:", len(df.columns))
print(df.columns.tolist())

Aantal kolommen: 63
['wedstrijd_datum', 'wedstrijd_tijd', 'thuis_team', 'uit_team', 'doelpunten_thuis_voltijd', 'doelpunten_uit_voltijd', 'resultaat_voltijd', 'doelpunten_thuis_halftijd', 'doelpunten_uit_halftijd', 'resultaat_halftijd', 'schoten_thuis', 'schoten_uit', 'schoten_op_doel_thuis', 'schoten_op_doel_uit', 'overtredingen_thuis', 'overtredingen_uit', 'corners_thuis', 'corners_uit', 'gele_kaarten_thuis', 'gele_kaarten_uit', 'rode_kaarten_thuis', 'rode_kaarten_uit', 'temperatuur_c', 'neerslag_mm', 'relatieve_luchtvochtigheid_pct', 'windsnelheid_m_s', 'windrichting_graden', 'windstoten_m_s', 'bewolking_pct', 'weercode', 'luchtdruk_hpa', 'temperatuur_gem_c', 'neerslag_som_mm', 'windsnelheid_gem_m_s', 'windstoten_max_m_s', 'temperatuur_gem_laatste48_c', 'neerslag_som_laatste48_mm', 'windstoten_max_laatste48_m_s', 'regen_uren_laatste48', 'hitte_uren_laatste48', 'thuis_team_naam', 'thuis_stadion_capaciteit', 'uit_team_naam', 'uit_stadion_capaciteit', 'seizoen', 'thuis_rank', 'uit_rank

### Verwijderen overbodige/gecorreleerde features

In [456]:
df = df.drop(['wedstrijd_tijd', 'thuis_team_naam', 'uit_team_naam', 'temperatuur_c', 'neerslag_mm', 'windsnelheid_m_s',
              'doelpunten_thuis_halftijd', 'doelpunten_uit_halftijd', 'resultaat_halftijd',
'temperatuur_gem_laatste48_c',
    'neerslag_som_laatste48_mm',
    'windstoten_max_laatste48_m_s',
    'regen_uren_laatste48',
    'hitte_uren_laatste48'], axis=1)

In [458]:
print("Aantal kolommen:", len(df.columns))
print(df.columns.tolist())

Aantal kolommen: 49
['wedstrijd_datum', 'thuis_team', 'uit_team', 'doelpunten_thuis_voltijd', 'doelpunten_uit_voltijd', 'resultaat_voltijd', 'schoten_thuis', 'schoten_uit', 'schoten_op_doel_thuis', 'schoten_op_doel_uit', 'overtredingen_thuis', 'overtredingen_uit', 'corners_thuis', 'corners_uit', 'gele_kaarten_thuis', 'gele_kaarten_uit', 'rode_kaarten_thuis', 'rode_kaarten_uit', 'relatieve_luchtvochtigheid_pct', 'windrichting_graden', 'windstoten_m_s', 'bewolking_pct', 'weercode', 'luchtdruk_hpa', 'temperatuur_gem_c', 'neerslag_som_mm', 'windsnelheid_gem_m_s', 'windstoten_max_m_s', 'thuis_stadion_capaciteit', 'uit_stadion_capaciteit', 'seizoen', 'thuis_rank', 'uit_rank', 'titel_match', 'degradatie_match', 'thuis_speelstijl', 'uit_speelstijl', 'thuis_form_5', 'uit_form_5', 'h2h_thuis_wins', 'weekday', 'avond_match', 'regen', 'harde_regen', 'warm', 'speelstijl_combo', 'is_weekend', 'rank_diff', 'form_diff']


## 5. Exploratory Data Analysis (EDA)

- Verdelen van thuis- en uitwinst
- Rangverschil versus resultaat
- Vormanalyse
- Weersomstandigheden
- Schoten / corners
- Grafieken en commentaar

## 6. Correlatieanalyse

- Factoren vs resultaten
- Factoren vs doelpunten
- Factoren vs kaarten
- Heatmaps of tabellen

## 7. Model voor wedstrijdresultaat

- Keuze model (logistische regressie, classificatie)
- Features en target
- Train-test split
- Resultaten evaluatie

## 8. Model voor doelpunten

- Keuze model (Poisson regression, regressie)
- Features en target
- Resultaten evaluatie

## 9. Model evaluatie

- Accuracy, log-loss, RMSE, MAE
- Grafieken van voorspelling vs echte resultaten

## 10. Expertdata importeren

- Inspectie expert dataset
- Beschrijving kolommen (probabilities, voorspelde doelpunten, confidence, motivatie)

## 11. Experts vs Model vergelijken

- Kansvoorspellingen (Brier score)
- Doelpunten (MAE / RMSE)
- Visualisatie (scatterplots, barplots)

## 12. Confidence vs Accuracy

- Confidence bins
- Accuracy per bin
- Calibration plots
- Analyse van over- of underconfidence

## 13. Bias analyse

- Thuisvoordeel overschatting
- Motivatie overschatting
- Vorm, rangverschil
- Histogrammen / visualisaties

## 14. Visualisaties

- Factor importance
- Experts vs model resultaten
- Doelpunten vergelijking
- Confidence calibration
- Interactieve grafieken (optioneel)

## 15. Conclusie / Discussie

- Belangrijkste bevindingen
- Factoren met grootste effect
- Verschillen experts vs model
- Limitaties
- Suggesties vervolgonderzoek